# Modul 6 – Filter Spasial
Low Pass Filter, High Pass Filter, Point Detection, Line Detection, Edge Detection


## D. Praktikum Filter


### Langkah 2.b — Akses Google Drive dan Import Library


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import math
from google.colab.patches import cv2_imshow
from PIL import Image as im


### Langkah 2.c — Membuat Fungsi Konvolusi Tanpa Library OpenCV


In [ ]:
def convolution2d(image, kernel, stride=1, padding=0, title=None):
    if padding > 0:
        image = np.pad(image, ((padding, padding), (padding, padding)), mode='constant', constant_values=0)
    
    kernel_height, kernel_width = kernel.shape
    image_height, image_width = image.shape
    
    output_height = int((image_height - kernel_height) / stride) + 1
    output_width = int((image_width - kernel_width) / stride) + 1
    
    output = np.zeros((output_height, output_width), dtype=np.float32)
    
    for y in range(output_height):
        for x in range(output_width):
            y_start = y * stride
            y_end = y_start + kernel_height
            x_start = x * stride
            x_end = x_start + kernel_width
            
            sub_image = image[y_start:y_end, x_start:x_end]
            output[y, x] = np.sum(sub_image * kernel)
            
    output = np.clip(output, 0, 255).astype(np.uint8)
    
    plt.figure()
    plt.imshow(output, cmap='gray')
    if title:
        plt.title(title)
    plt.show()
    
    return output


### Langkah 2.d — Load Citra Masukan dan Konversi ke Citra Keabuan (Grayscale)


In [ ]:
img = cv.imread('/content/drive/MyDrive/Polinema/Kuliah/PCVK/Images/mandrill.tiff')
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)


### Langkah 2.e — Menentukan Kernel Sharpening


In [ ]:
# image sharpen
kernel_sharpen = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]])


### Langkah 2.f — Memanggil Fungsi Konvolusi dan Menampilkan Hasil Sharpening


In [ ]:
convolution2d(img_gray, kernel_sharpen, 1, 2, title="Sharpen")


### Langkah 3 — Penerapan Berbagai Kernel Filter Spasial Menggunakan convolution2d


#### 1. Sharpen Filter


In [ ]:
kernel_sharpen = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]], dtype=np.float32)
convolution2d(img_gray, kernel_sharpen, 1, 1, title="Sharpen")


#### 2. Emboss Filter


In [ ]:
kernel_emboss = np.array([[-2, -1, 0],
                          [-1,  1, 1],
                          [ 0,  1, 2]], dtype=np.float32)
convolution2d(img_gray, kernel_emboss, 1, 1, title="Emboss")


#### 3. Left Sobel Edge Detection


In [ ]:
kernel_left_sobel = np.array([[1, 0, -1],
                              [2, 0, -2],
                              [1, 0, -1]], dtype=np.float32)
convolution2d(img_gray, kernel_left_sobel, 1, 1, title="Left Sobel Edge Detection")


#### 4. Canny Edge Detection (Laplacian / Edge Detection Kernel)


In [ ]:
kernel_canny = np.array([[-1, -1, -1],
                          [-1,  8, -1],
                          [-1, -1, -1]], dtype=np.float32)
convolution2d(img_gray, kernel_canny, 1, 1, title="Canny Edge Detection")


#### 5. 21x21 Gaussian Blur


In [ ]:
kernel_size = 21
sigma = math.sqrt(kernel_size)
gaussian_kernel = cv.getGaussianKernel(kernel_size, sigma)
gauss_kernel = gaussian_kernel @ gaussian_kernel.transpose()

convolution2d(img_gray, gauss_kernel, 1, kernel_size // 2, title="21x21 Gaussian Blur")


#### 6. Average Filter 3x3


In [ ]:
kernel_average = (1.0 / 9.0) * np.ones((3, 3), dtype=np.float32)
convolution2d(img_gray, kernel_average, 1, 1, title="Average Filter")


#### 7. Low Pass Filter


In [ ]:
kernel_low_pass = (1.0 / 12.0) * np.array([
    [1, 1, 1],
    [1, 4, 1],
    [1, 1, 1]
], dtype=np.float32)
convolution2d(img_gray, kernel_low_pass, 1, 1, title="Low Pass Filter")


#### 8. High Pass Filter


In [ ]:
kernel_high_pass = np.array([
    [-1, 0, 1],
    [-1, 0, 3],
    [-3, 0, 1]
], dtype=np.float32)
convolution2d(img_gray, kernel_high_pass, 1, 1, title="High Pass Filter")


## E. Filter Library dan Filter Modern


### Percobaan 1: Filter Gaussian, Sharpen, dan Canny Menggunakan Library filter2D OpenCV


In [ ]:
# Fungsi tampil berdampingan
def show_side_by_side(images, titles, figsize=(15,5)):
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2: # grayscale
            plt.subplot(1, len(images), i+1)
            plt.imshow(img, cmap="gray")
        else: # color
            plt.subplot(1, len(images), i+1)
            plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis("off")
    plt.show()

img = cv.imread("/content/drive/MyDrive/Kuliah/PCVK Genap 2023/Images/lena.jpg")
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

blur = cv.GaussianBlur(img, (7,7), 1)
edges = cv.Canny(cv.cvtColor(img, cv.COLOR_BGR2GRAY), 100, 200)
sharpen_kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
sharpened = cv.filter2D(img, -1, sharpen_kernel)
show_side_by_side([img, blur, sharpened, edges],
                  ["Asli", "Gaussian Blur", "Sharpened", "Canny Edge Detection"])


### Percobaan 2: Filter Modern (Bilateral Filtering dan Edge Preserving Filter)


In [ ]:
#Filter Modern dari OpenCV
# Bilateral Filter (edge-preserving)
bilateral = cv.bilateralFilter(img, 50, 100, 100)

# Edge Preserving Filter (alternatif Guided Filter)
edge_preserve = cv.edgePreservingFilter(img, flags=1, sigma_s=100, sigma_r=0.9)

show_side_by_side([img, bilateral, edge_preserve],
                  ["Asli", "Bilateral Filter", "Edge Preserving Filter"])


### Percobaan 3: Filtering Feature Map pada CNN


In [ ]:
#Filter Feature Map yang digunakan pada CNN, Lakukan running code bagian ini beberapa kali dan perhatikan hasilnya
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        return self.conv1(x)

model = SimpleCNN()

# Ubah gambar ke tensor
img_tensor = torch.tensor(img_gray, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0

# Hasil CNN
with torch.no_grad():
    features = model(img_tensor)

# Visualisasi feature maps
feature_maps = [features[0,i].numpy() for i in range(features.shape[1])]
show_side_by_side([img_gray] + feature_maps, ["Asli (Gray)"] + [f"Feature {i+1}" for i in range(len(feature_maps))])


**Kesimpulan Percobaan 3:**

Setiap filter/kernel konvolusi pada layer convolutional CNN (`Conv2d`) yang diinisialisasi secara acak menghasilkan feature map yang berbeda-beda. Beberapa filter mengekstrak informasi tepi (horizontal/vertikal), tekstur, maupun kontras lokal dari citra masukan. Pada model CNN terarah yang dilatih, bobot-bobot kernel ini akan dioptimalkan secara otomatis melalui proses backpropagation untuk menangkap representasi fitur visual yang paling relevan dengan tugas tertentu.


### Percobaan 4: Efek Beauty dan Old/Vintage Filter


In [ ]:
# =====================
# 1. Beauty Filter
# =====================
# Step 1: Smoothing kulit dengan bilateral filter
smooth = cv.bilateralFilter(img, d=15, sigmaColor=75, sigmaSpace=75)

# Step 2: Unsharp masking (pertajam mata/bibir)
gaussian = cv.GaussianBlur(smooth, (0,0), 3)
sharpened = cv.addWeighted(smooth, 1.5, gaussian, -0.5, 0)

# Step 3: Brightness & contrast
alpha = 1.2  # contrast
beta = 15    # brightness
beauty = cv.convertScaleAbs(sharpened, alpha=alpha, beta=beta)

# =====================
# 2. Old/Vintage Filter
# =====================
# Step 1: Sepia tone
sepia_kernel = np.array([[0.272, 0.534, 0.131],
                         [0.349, 0.686, 0.168],
                         [0.393, 0.769, 0.189]])
sepia = cv.transform(img, sepia_kernel)
sepia = np.clip(sepia, 0, 255).astype(np.uint8)

# Step 2: Vignette
rows, cols = img.shape[:2]
kernel_x = cv.getGaussianKernel(cols, cols*0.6)
kernel_y = cv.getGaussianKernel(rows, rows*0.6)
kernel = kernel_y * kernel_x.T
mask = kernel / kernel.max()
vignette = np.copy(sepia)
for i in range(3):
    vignette[:,:,i] = vignette[:,:,i] * mask

# Step 3: Noise/Grain
noise = np.random.normal(0, 15, vignette.shape).astype(np.int16)
old_img = np.clip(vignette.astype(np.int16) + noise, 0, 255).astype(np.uint8)

show_side_by_side([img, beauty, old_img], ["Asli", "Beauty Filter", "Old/Vintage Filter"])


### Percobaan 5: Filter Anime / Cartoon


In [ ]:
#Filter Anime / Cartoon
# Step 1: Edge detection (pakai median blur dulu agar gambar menjadi lebih halus)
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
gray_blur = cv.medianBlur(gray, 7)
edges = cv.adaptiveThreshold(gray_blur, 255,
                             cv.ADAPTIVE_THRESH_MEAN_C,
                             cv.THRESH_BINARY, 9, 9)

# Step 2: Bilateral filter untuk smoothing warna
color = cv.bilateralFilter(img, d=9, sigmaColor=200, sigmaSpace=200)

# Step 3: Gabungkan (cartoonize)
cartoon = cv.bitwise_and(color, color, mask=edges)

# Tampilkan
show_side_by_side([img, cartoon], ["Asli", "Cartoon Filter"])


### Percobaan 6: Night Filter


In [ ]:
#Night Filter
img = cv.imread("/content/drive/MyDrive/Kuliah/PCVK Genap 2023/Images/djawatan.jpg")
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

# Step 1: Gelapkan (contrast turun, brightness negatif)
night = cv.convertScaleAbs(img, alpha=0.6, beta=-40)

# Step 2: Tambah bias biru
blue_tint = np.full_like(night, (50, 0, 100)) # BGR
night = cv.addWeighted(night, 0.8, blue_tint, 0.2, 0)

# Step 3: Efek glow di area terang dengan filter2D (blur kernel)
kernel = np.ones((15,15), np.float32) / 225
glow = cv.filter2D(night, -1, kernel)

# Kombinasikan asli + glow
night_glow = cv.addWeighted(night, 0.7, glow, 0.3, 0)

show_side_by_side([img, night, night_glow],
                  ["Asli", "Night Filter", "Night + Glow (filter2D)"])


### Percobaan 7: Filter Suasana Pagi dan Kabut


In [ ]:
#Filter Suasana pagi dan Kabut
# ========================
# Step 1: Kurangi kontras & cerahkan
# ========================
alpha = 0.9  # contrast
beta = 20    # brightness
soft = cv.convertScaleAbs(img, alpha=alpha, beta=beta)

# ========================
# Step 2: Tambahkan warm tone (kemerahan / oranye)
# ========================
warm_tint = np.full_like(soft, (40, 70, 120))  # BGR
pagi = cv.addWeighted(soft, 0.8, warm_tint, 0.2, 0)

# ========================
# Step 3: Tambahkan haze (kabut tipis) dengan filter2D
# ========================
# Kernel blur Gaussian-like untuk menciptakan efek kabut
kernel = cv.getGaussianKernel(3, 3)
kernel = kernel @ kernel.T  # jadikan 2D kernel
kabut = cv.filter2D(pagi, -1, kernel)

# tambah layer putih untuk kabut lebih nyata
white_layer = np.full_like(pagi, 255)
kabut = cv.addWeighted(kabut, 0.7, white_layer, 0.3, 0)

show_side_by_side([img, pagi, kabut],
                  ["Asli", "Pagi", "Pagi + Kabut"])


## Tugas Praktikum


### 1. Tugas Analisis : Perbandingan Mean Filter (Blur Biasa) vs. Median Filter pada Noise "Salt and Pepper"

a. Tambahkan noise bintik hitam-putih (Salt and Pepper) ke citra grayscale.
b. Terapkan Mean Filter (3 x 3) dan Median Filter (3 x 3).
c. Analisis visual perbandingan Mean Filter vs. Median Filter.


In [ ]:
# Muat citra pengujian untuk Tugas Praktikum
img = cv.imread("/content/drive/MyDrive/Kuliah/PCVK Genap 2023/Images/lena.jpg")
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

def add_salt_and_pepper_noise(image, salt_prob=0.02, pepper_prob=0.02):
    noisy = image.copy()
    # Salt (bintik putih = 255)
    num_salt = int(np.ceil(salt_prob * image.size))
    coords_salt = [np.random.randint(0, i, num_salt) for i in image.shape]
    noisy[tuple(coords_salt)] = 255
    
    # Pepper (bintik hitam = 0)
    num_pepper = int(np.ceil(pepper_prob * image.size))
    coords_pepper = [np.random.randint(0, i, num_pepper) for i in image.shape]
    noisy[tuple(coords_pepper)] = 0
    return noisy

# a. Tambahkan noise bintik hitam-putih (Salt and Pepper)
noisy_img = add_salt_and_pepper_noise(img_gray, salt_prob=0.03, pepper_prob=0.03)

# b. Terapkan Mean Filter (3 x 3) dan Median Filter (3 x 3)
mean_filtered = cv.blur(noisy_img, (3, 3))
median_filtered = cv.medianBlur(noisy_img, 3)

show_side_by_side([noisy_img, mean_filtered, median_filtered],
                  ["Citra Noisy (Salt & Pepper)", "Mean Filter (3x3)", "Median Filter (3x3)"])


**c. Analisis Visual Perbandingan Mean Filter vs. Median Filter:**

- **Mean Filter (Linear)**: Bekerja dengan menjumlahkan nilai seluruh piksel tetangga dalam jendela kernel dan menghitung nilai rata-ratanya. Karena noise salt (255) dan pepper (0) memiliki nilai intensitas ekstrem, nilai ekstrem ini ikut terserap ke dalam perhitungan rata-rata. Akibatnya, noise tidak hilang melainkan menyebar menjadi bercak buram kelabu di sekitar piksel asal, serta menyebabkan penurunan ketajaman tepi citra secara signifikan.
- **Median Filter (Non-Linear)**: Bekerja dengan mengurutkan nilai piksel tetangga dalam jendela kernel lalu memilih nilai tengah (median). Nilai derau salt dan pepper yang berada di ujung spektrum ekstrem secara otomatis tersingkir dan tidak pernah terpilih sebagai nilai median. Oleh karena itu, Median Filter jauh lebih bersih dan efektif dalam melenyapkan derau salt & pepper tanpa mengaburkan detail tepi objek.


### 2. Tugas Evaluasi: Pemilihan Ukuran Kernel terbaik untuk Menajamkan Citra (Sharpening)

a. Buat kernel penajaman ukuran 3 x 3 dan 5 x 5.
b. Terapkan pada citra yang sedikit buram (blurry).
c. Evaluasi aspek kejelasan tepi vs. kadar noise dan pemilihan kernel optimal.


In [ ]:
# a. Buat kernel penajaman (Sharpening Kernel) ukuran 3 x 3 dan 5 x 5
kernel_sharpen_3x3 = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype=np.float32)

kernel_sharpen_5x5 = np.array([
    [-1, -1, -1, -1, -1],
    [-1, -1, -1, -1, -1],
    [-1, -1, 25, -1, -1],
    [-1, -1, -1, -1, -1],
    [-1, -1, -1, -1, -1]
], dtype=np.float32)

# b. Terapkan pada citra yang sedikit buram (blurry)
blurry_img = cv.GaussianBlur(img_gray, (5, 5), 1)

sharpened_3x3 = cv.filter2D(blurry_img, -1, kernel_sharpen_3x3)
sharpened_5x5 = cv.filter2D(blurry_img, -1, kernel_sharpen_5x5)

show_side_by_side([blurry_img, sharpened_3x3, sharpened_5x5],
                  ["Citra Agak Buram", "Sharpening 3x3", "Sharpening 5x5"])


**c. Evaluasi Hasil Berdasarkan Aspek Kejelasan Tepi vs. Kadar Noise:**

- **Kernel 3x3**: Memberikan peningkatan kontras tepi secara proporsional dan natural. Garis batas objek menjadi lebih tegas tanpa memicu artefak distorsi atau memperbesar noise pada area yang relatif halus.
- **Kernel 5x5**: Menghasilkan efek penajaman yang berlebihan (over-sharpening). Menimbulkan artefak halo (cincin putih/hitam di sepanjang tepi kontras tinggi) serta mengamplifikasi noise frekuensi tinggi secara agresif, membuat tekstur citra terlihat kasar dan artifisial.
- **Kesimpulan Kernel Optimal**: Kernel **3x3** adalah pilihan paling optimal untuk penggunaan sehari-hari karena mempertahankan keseimbangan terbaik antara ketajaman tepi yang jelas dan distorsi noise yang minimal.


### 3. Tugas Kreasi: Filter Kartun Sederhana

a. Buatlah fungsi kustom bernama `kartun(image)` yang menggabungkan Bilateral Filter, deteksi tepi (Adaptive Thresholding/Canny), dan penggabungan secara logika.
b. Tampilkan hasil citra asli dengan citra efek kartun bersebelahan.


In [ ]:
# a. Fungsi kustom kartun(image)
def kartun(image):
    # 1. Penghalusan warna menggunakan Bilateral Filter agar area warna menjadi rata
    color = cv.bilateralFilter(image, d=9, sigmaColor=200, sigmaSpace=200)
    
    # 2. Deteksi garis tepi menggunakan Adaptive Thresholding
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    gray_blur = cv.medianBlur(gray, 7)
    edges = cv.adaptiveThreshold(
        gray_blur, 255,
        cv.ADAPTIVE_THRESH_MEAN_C,
        cv.THRESH_BINARY, 9, 9
    )
    
    # 3. Penggabungan (kombinasi) keduanya menggunakan operasi logika (bitwise_and)
    cartoon_img = cv.bitwise_and(color, color, mask=edges)
    
    return cartoon_img

# b. Tampilkan hasil citra asli dengan citra efek kartun bersebelahan
img_cartoon = kartun(img)
show_side_by_side([img, img_cartoon], ["Citra Asli", "Efek Kartun (kartun())"])
